In [51]:
# 필요한 패키지
# pip install sqlalchemy pymysql pandas

import pandas as pd
from sqlalchemy import create_engine, text

# ── DB 접속 정보 ───────────────────────────────────────────────────────────────
from DATA.stock_invest_function import get_db_host

db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

TABLE_NAME = "Korea_company_valuation_ver2"  # 스키마: investar.TABLE_NAME

def make_engine(db):
    url = (
        f"mysql+pymysql://{db['user']}:{db['password']}"
        f"@{db['host']}:{db['port']}/{db['database']}?charset=utf8mb4"
    )
    return create_engine(url, pool_pre_ping=True, future=True)

engine = make_engine(db_info)

# 1) forecast_date의 unique 값 추출
def get_unique_forecast_dates(include_null=False):
    q = f"""
        SELECT DISTINCT forecast_date
        FROM {TABLE_NAME}
        {"WHERE forecast_date IS NOT NULL" if not include_null else ""}
        ORDER BY forecast_date
    """
    with engine.begin() as conn:
        df = pd.read_sql(q, conn, parse_dates=["forecast_date"])
    return df["forecast_date"]

# 2) (ticker, forecast_date, keyword)로 indicator에 keyword가 포함된 값 조회 + date 기준 정렬
#    여러 indicator가 매칭되면 행으로 반환(롱 포맷). wide=True면 indicator별 칼럼으로 피벗.
def get_series_by_keyword(ticker, forecast_date, keyword, wide=False):
    sql = text(f"""
        SELECT `date`, `ticker`, `indicator`, `value`, `forecast_date`
        FROM {TABLE_NAME}
        WHERE ticker = :ticker
          AND forecast_date = :fdate
          AND indicator LIKE :kw
        ORDER BY `date`
    """)
    with engine.begin() as conn:
        df = pd.read_sql(
            sql, conn,
            params={"ticker": ticker, "fdate": forecast_date, "kw": f"%{keyword}%"},
            parse_dates=["date", "forecast_date"]
        )
    # 숫자형 보정
    if not df.empty:
        df["value"] = pd.to_numeric(df["value"], errors="coerce")
    if wide and not df.empty:
        df_wide = df.pivot_table(index="date", columns="indicator", values="value", aggfunc="last").sort_index()
        df_wide = df_wide.rename_axis(None, axis=1)
        return df_wide
    return df  # 롱 포맷: date, indicator, value …

# 3) indicator의 unique 값 추출
def get_unique_indicators(keyword=None):
    cond = "" if not keyword else "WHERE indicator LIKE :kw"
    sql = text(f"SELECT DISTINCT indicator FROM {TABLE_NAME} {cond} ORDER BY indicator")
    with engine.begin() as conn:
        df = pd.read_sql(sql, conn, params=(None if not keyword else {"kw": f"%{keyword}%"}))
    return df["indicator"]

# 4) (ticker, indicator, forecast_date 두 개) 입력 시 두 기간 차이 비교
#    반환: date 기준 병합(outer), col: value_fd1, value_fd2, diff = fd2 - fd1
def compare_indicator_between_dates(ticker, indicator, forecast_date_1, forecast_date_2):
    base_sql = text(f"""
        SELECT `date`, `value`
        FROM {TABLE_NAME}
        WHERE ticker = :ticker
          AND indicator = :indicator
          AND forecast_date = :fdate
        ORDER BY `date`
    """)
    with engine.begin() as conn:
        df1 = pd.read_sql(
            base_sql, conn,
            params={"ticker": ticker, "indicator": indicator, "fdate": forecast_date_1},
            parse_dates=["date"]
        )
        df2 = pd.read_sql(
            base_sql, conn,
            params={"ticker": ticker, "indicator": indicator, "fdate": forecast_date_2},
            parse_dates=["date"]
        )

    # 숫자형 보정
    for d in (df1, df2):
        if not d.empty:
            d["value"] = pd.to_numeric(d["value"], errors="coerce")

    df1 = df1.rename(columns={"value": f"value_{pd.to_datetime(forecast_date_1).date()}"})
    df2 = df2.rename(columns={"value": f"value_{pd.to_datetime(forecast_date_2).date()}"})

    out = pd.merge(df1, df2, on="date", how="outer").sort_values("date").set_index("date")
    if out.shape[1] == 2:
        cols = out.columns.tolist()
        out["diff"] = out[cols[1]] - out[cols[0]]  # fd2 - fd1
    return out

# -*- coding: utf-8 -*-
from __future__ import annotations
import pandas as pd
from typing import Optional, Union, Sequence, Tuple
from sqlalchemy import create_engine, text

# ─────────────────────────────────────────────────────────────────────────────

def make_engine(db_info: dict):
    url = (
        "mysql+pymysql://{user}:{password}@{host}:{port}/{database}"
        "?charset=utf8mb4"
    ).format(**db_info)
    return create_engine(url, pool_recycle=3600, pool_pre_ping=True)


# ─────────────────────────────────────────────────────────────────────────────
# 1) 특정 forecast_date로 유니크 티커 조회
#   - forecast_date가 None → forecast_date IS NULL 조건
#   - forecast_date가 'YYYY-MM-DD' 나 'YYYY-MM-DD HH:MM:SS' → 해당 일자/시각 매칭
#   - '같은 날'로 묶고 싶다면 use_date_only=True 로 DATE(forecast_date)=DATE(:dt)
# ─────────────────────────────────────────────────────────────────────────────
def get_unique_tickers_by_forecast_date(
    db_info: dict,
    forecast_date: Optional[str] = None,
    table_name: str = "valuation_forecast_result",
    use_date_only: bool = True,
) -> pd.Series:
    """
    Return: pd.Series of unique tickers (name='ticker')
    """
    engine = make_engine(db_info)
    with engine.connect() as conn:
        if forecast_date is None:
            sql = text(f"""
                SELECT DISTINCT ticker
                FROM {table_name}
                WHERE forecast_date IS NULL
                ORDER BY ticker
            """)
            df = pd.read_sql(sql, conn)
        else:
            if use_date_only:
                sql = text(f"""
                    SELECT DISTINCT ticker
                    FROM {table_name}
                    WHERE DATE(forecast_date) = DATE(:dt)
                    ORDER BY ticker
                """)
            else:
                sql = text(f"""
                    SELECT DISTINCT ticker
                    FROM {table_name}
                    WHERE forecast_date = :dt
                    ORDER BY ticker
                """)
            df = pd.read_sql(sql, conn, params={"dt": forecast_date})

    return df["ticker"]


# ─────────────────────────────────────────────────────────────────────────────
# 2) indicator별 date1→date2 변화율 계산
#   - indicator: str 또는 [str, ...]  (None/'all'은 전체)
#   - 변화율 = (v2 / v1 - 1).  v1=0 또는 결측은 안전하게 제외
#   - 결과 정렬: pct_change(%) 내림차순
# Columns:
#   ['indicator','ticker','date1','date2','value_date1','value_date2',
#    'abs_change','pct_change']
# ─────────────────────────────────────────────────────────────────────────────
def get_indicator_change_rates(
    db_info: dict,
    forecast_date: Optional[str],
    indicator: Optional[Union[str, Sequence[str]]] = None,
    date1: str = "2027-03-31",
    date2: str = "2027-06-30",
    table_name: str = "valuation_forecast_result",
    use_date_only_for_forecast: bool = True,
    drop_zero_base: bool = True,
) -> pd.DataFrame:
    # ... (위쪽 동일: indicator_list 정리, where 조건 빌드) ...
    engine = make_engine(db_info)
    with engine.connect() as conn:
        conds = []
        params = {}

        # forecast_date filter
        if forecast_date is None:
            conds.append("forecast_date IS NULL")
        else:
            if use_date_only_for_forecast:
                conds.append("DATE(forecast_date) = DATE(:fdt)")
            else:
                conds.append("forecast_date = :fdt")
            params["fdt"] = forecast_date

        # indicator filter
        if indicator is None or (isinstance(indicator, str) and indicator.lower() == "all"):
            pass
        else:
            if isinstance(indicator, str):
                indicator_list = [indicator]
            else:
                indicator_list = list(indicator)
            placeholders = []
            for i, it in enumerate(indicator_list):
                key = f"i{i}"
                params[key] = it
                placeholders.append(f":{key}")
            conds.append(f"indicator IN ({', '.join(placeholders)})")

        # date filter
        conds.append("DATE(date) IN (DATE(:d1), DATE(:d2))")
        params["d1"] = date1
        params["d2"] = date2

        where_sql = " AND ".join(conds)

        # ⭐️ value가 문자일 수 있어 REPLACE 후 CAST: '1,234.56' -> 1234.56
        sql = text(f"""
            SELECT
                DATE(date) AS d,
                ticker,
                indicator,
                CAST(REPLACE(value, ',', '') AS DECIMAL(38, 8)) AS value
            FROM {table_name}
            WHERE {where_sql}
        """)
        raw = pd.read_sql(sql, conn, params=params)

    if raw.empty:
        return pd.DataFrame(columns=[
            "indicator","ticker","date1","date2",
            "value_date1","value_date2","abs_change","pct_change"
        ])

    # pivot
    piv = (
        raw
        .assign(d=lambda x: pd.to_datetime(x["d"]).dt.date)
        .pivot_table(index=["indicator","ticker"], columns="d", values="value", aggfunc="last")
        .reset_index()
    )

    d1 = pd.to_datetime(date1).date()
    d2 = pd.to_datetime(date2).date()
    if d1 not in piv.columns: piv[d1] = pd.NA
    if d2 not in piv.columns: piv[d2] = pd.NA

    piv = piv.rename(columns={d1: "value_date1", d2: "value_date2"})
    out = piv[["indicator","ticker","value_date1","value_date2"]].copy()

    # ⭐️ 혹시 남아있는 문자열/공백/퍼센트 대비 2차 방어
    def _to_numeric_safe(s: pd.Series) -> pd.Series:
        # 문자열이면 콤마, % 제거 후 숫자 변환
        if s.dtype == "object":
            s = s.astype(str).str.replace(",", "", regex=False).str.replace("%", "", regex=False).str.strip()
        return pd.to_numeric(s, errors="coerce")

    out["value_date1"] = _to_numeric_safe(out["value_date1"])
    out["value_date2"] = _to_numeric_safe(out["value_date2"])

    # 결측/분모 0 제거
    out = out.dropna(subset=["value_date1","value_date2"])
    if drop_zero_base:
        out = out[out["value_date1"] != 0]

    # 변화/변화율
    out["abs_change"] = out["value_date2"] - out["value_date1"]
    out["pct_change"] = (out["value_date2"] / out["value_date1"] - 1.0) * 100.0

    # 메타/정렬
    out.insert(2, "date1", d1)
    out.insert(3, "date2", d2)
    out = out.sort_values(["indicator","pct_change"], ascending=[True, False]).reset_index(drop=True)

    return out



# ─────────────────────────────────────────────────────────────────────────────
# 사용 예시
# ─────────────────────────────────────────────────────────────────────────────
# if __name__ == "__main__":
#     db_info = {
#         "host": "127.0.0.1",
#         "port": 3307,
#         "user": "stox7412",
#         "password": "*****",
#         "database": "investar",
#     }
#
#     # 1) 특정 forecast_date의 유니크 티커
#     #    NULL 날짜 기준이라면 forecast_date=None
#     tickers = get_unique_tickers_by_forecast_date(
#         db_info=db_info,
#         forecast_date=None,  # 예: NULL rows
#         table_name="valuation_forecast_result",
#         use_date_only=True,
#     )
#     print("[Unique tickers]\n", tickers.head())
#
#     # 2) 변화율 표 (예: forecast_date가 NULL, 모든 indicator, 2027-03-31 → 2027-06-30)
#     df_change = get_indicator_change_rates(
#         db_info=db_info,
#         forecast_date=None,                    # 또는 '2027-06-30' 등
#         indicator=None,                        # 'all' 또는 ['sarima_forecast','lstm_forecast'] 등
#         date1="2027-03-31",
#         date2="2027-06-30",
#         table_name="valuation_forecast_result",
#         use_date_only_for_forecast=True,
#         drop_zero_base=True,
#     )
#     print("\n[Change table]\n", df_change.head(20))



# ── 사용 예시 ─────────────────────────────────────────────────────────────────
# if __name__ == "__main__":
#     # 1) forecast_date 목록
#     print(get_unique_forecast_dates().tail())
#
#     # 2) 키워드로 조회 (롱/와이드)
#     ex_long = get_series_by_keyword(ticker="A005930", forecast_date="2025-10-26", keyword="revenue", wide=False)
#     ex_wide = get_series_by_keyword(ticker="A005930", forecast_date="2025-10-26", keyword="revenue", wide=True)
#     print(ex_long.head())
#     print(ex_wide.head())
#
#     # 3) indicator 유니크
#     print(get_unique_indicators().head())
#     # 특정 키워드만
#     print(get_unique_indicators(keyword="forecast").head())
#
#     # 4) 두 forecast_date 비교
#     comp = compare_indicator_between_dates(
#         ticker="A005930",
#         indicator="revenue_ensemble_forecast",   # 예: 정확한 indicator 이름 입력
#         forecast_date_1="2025-10-26",
#         forecast_date_2="2025-10-29"
#     )
#     print(comp.tail())



In [2]:
print(get_unique_forecast_dates().tail())

0   2025-10-26
1   2025-10-29
2   2025-10-30
3   2025-10-31
Name: forecast_date, dtype: datetime64[ns]


In [69]:
ex_long = get_series_by_keyword(ticker="A000660", forecast_date="2025-10-31", keyword="rev", wide=False)

In [70]:
ex_long.tail(15)

,date,ticker,indicator,value,forecast_date
1059,2026-06-30,A000660,revenue_prophet_ttm,6.296568e+10,2025-10-31
1060,2026-06-30,A000660,revenue_lstm_ttm,5.714418e+10,2025-10-31
1061,2026-06-30,A000660,revenue_theta_ttm,9.031895e+10,2025-10-31
1062,2026-09-30,A000660,revenue_sarima,1.639958e+10,2025-10-31
1063,2026-09-30,A000660,revenue_sarima_exog,2.837641e+10,2025-10-31
1064,2026-09-30,A000660,revenue_ets,2.532085e+10,2025-10-31
1065,2026-09-30,A000660,revenue_prophet,1.643817e+10,2025-10-31
1066,2026-09-30,A000660,revenue_lstm,1.610055e+10,2025-10-31
1067,2026-09-30,A000660,revenue_theta,2.292816e+10,2025-10-31
1068,2026-09-30,A000660,revenue_sarima_ttm,7.378905e+10,2025-10-31


In [71]:
# ex_long → indicator를 컬럼으로 피벗
ex_pivot = (
    ex_long
    .pivot_table(
        index=["date", "ticker"],      # 행 인덱스
        columns="indicator",           # 열로 변환할 컬럼
        values="value",                # 값으로 쓸 컬럼
        aggfunc="last"                 # 중복 시 마지막 값 사용
    )
    .reset_index()                     # date, ticker를 일반 컬럼으로 되돌림
)

# 필요 시 indicator 컬럼명 정리 (MultiIndex 제거)
ex_pivot.columns.name = None

# 확인
print(ex_pivot.head())


        date   ticker   revenue_ets  revenue_ets_ttm  revenue_lstm  \
0 2004-03-31  A000660  1.296647e+09              NaN  1.296647e+09   
1 2004-06-30  A000660  1.683512e+09              NaN  1.683512e+09   
2 2004-09-30  A000660  1.542350e+09              NaN  1.542350e+09   
3 2004-12-31  A000660  1.341844e+09     5.864353e+09  1.341844e+09   
4 2005-03-31  A000660  1.284277e+09     5.851983e+09  1.284277e+09   

   revenue_lstm_ttm  revenue_prophet  revenue_prophet_ttm  revenue_sarima  \
0               NaN     1.296647e+09                  NaN    1.296647e+09   
1               NaN     1.683512e+09                  NaN    1.683512e+09   
2               NaN     1.542350e+09                  NaN    1.542350e+09   
3      5.864353e+09     1.341844e+09         5.864353e+09    1.341844e+09   
4      5.851983e+09     1.284277e+09         5.851983e+09    1.284277e+09   

   revenue_sarima_exog  revenue_sarima_exog_ttm  revenue_sarima_ttm  \
0         1.296647e+09                      N

In [72]:
ex_pivot.tail(16)

,date,ticker,revenue_ets,revenue_ets_ttm,revenue_lstm,revenue_lstm_ttm,revenue_prophet,revenue_prophet_ttm,revenue_sarima,revenue_sarima_exog,revenue_sarima_exog_ttm,revenue_sarima_ttm,revenue_theta,revenue_theta_ttm
75,2022-12-31,A000660,7.672031e+09,4.462157e+10,7.672031e+09,4.462157e+10,7.672031e+09,4.462157e+10,7.672031e+09,7.672031e+09,4.462157e+10,4.462157e+10,7.672031e+09,4.462157e+10
76,2023-03-31,A000660,5.088111e+09,3.755403e+10,5.088111e+09,3.755403e+10,5.088111e+09,3.755403e+10,5.088111e+09,5.088111e+09,3.755403e+10,3.755403e+10,5.088111e+09,3.755403e+10
77,2023-06-30,A000660,7.305933e+09,3.104896e+10,7.305933e+09,3.104896e+10,7.305933e+09,3.104896e+10,7.305933e+09,7.305933e+09,3.104896e+10,3.104896e+10,7.305933e+09,3.104896e+10
78,2023-09-30,A000660,9.066171e+09,2.913225e+10,9.066171e+09,2.913225e+10,9.066171e+09,2.913225e+10,9.066171e+09,9.066171e+09,2.913225e+10,2.913225e+10,9.066171e+09,2.913225e+10
79,2023-12-31,A000660,1.130550e+10,3.276572e+10,1.130550e+10,3.276572e+10,1.130550e+10,3.276572e+10,1.130550e+10,1.130550e+10,3.276572e+10,3.276572e+10,1.130550e+10,3.276572e+10
80,2024-03-31,A000660,1.242960e+10,4.010721e+10,1.242960e+10,4.010721e+10,1.242960e+10,4.010721e+10,1.242960e+10,1.242960e+10,4.010721e+10,4.010721e+10,1.242960e+10,4.010721e+10
81,2024-06-30,A000660,1.642326e+10,4.922453e+10,1.642326e+10,4.922453e+10,1.642326e+10,4.922453e+10,1.642326e+10,1.642326e+10,4.922453e+10,4.922453e+10,1.642326e+10,4.922453e+10
82,2024-09-30,A000660,1.757307e+10,5.773143e+10,1.757307e+10,5.773143e+10,1.757307e+10,5.773143e+10,1.757307e+10,1.757307e+10,5.773143e+10,5.773143e+10,1.757307e+10,5.773143e+10
83,2024-12-31,A000660,1.976704e+10,6.619296e+10,1.976704e+10,6.619296e+10,1.976704e+10,6.619296e+10,1.976704e+10,1.976704e+10,6.619296e+10,6.619296e+10,1.976704e+10,6.619296e+10
84,2025-03-31,A000660,1.763914e+10,7.140250e+10,1.763914e+10,7.140250e+10,1.763914e+10,7.140250e+10,1.763914e+10,1.763914e+10,7.140250e+10,7.140250e+10,1.763914e+10,7.140250e+10


In [39]:
psr_long = get_series_by_keyword(ticker="A000660", forecast_date="2025-10-31", keyword="psr", wide=False)

# ex_long → indicator를 컬럼으로 피벗
psr_pivot = (
    psr_long
    .pivot_table(
        index=["date", "ticker"],      # 행 인덱스
        columns="indicator",           # 열로 변환할 컬럼
        values="value",                # 값으로 쓸 컬럼
        aggfunc="last"                 # 중복 시 마지막 값 사용
    )
    .reset_index()                     # date, ticker를 일반 컬럼으로 되돌림
)

# 필요 시 indicator 컬럼명 정리 (MultiIndex 제거)
psr_pivot.columns.name = None

# 확인
print(psr_pivot.head())

        date   ticker       psr  psr_ETS  psr_LSTM  psr_Prophet  \
0 2015-05-31  A000660  2.605408      NaN       NaN          NaN   
1 2015-06-30  A000660  2.156728      NaN       NaN          NaN   
2 2015-07-31  A000660  1.891599      NaN       NaN          NaN   
3 2015-08-31  A000660  1.784510      NaN       NaN          NaN   
4 2015-09-30  A000660  1.672355      NaN       NaN          NaN   

   psr_SARIMA_exog  psr_SARIMA_noexog  psr_Theta  
0              NaN                NaN        NaN  
1              NaN                NaN        NaN  
2              NaN                NaN        NaN  
3              NaN                NaN        NaN  
4              NaN                NaN        NaN  


In [40]:
psr_pivot.tail(20)

,date,ticker,psr,psr_ETS,psr_LSTM,psr_Prophet,psr_SARIMA_exog,psr_SARIMA_noexog,psr_Theta
119,2025-04-30,A000660,2.403496,NaN,NaN,NaN,NaN,NaN,NaN
120,2025-05-31,A000660,2.707858,NaN,NaN,NaN,NaN,NaN,NaN
121,2025-06-30,A000660,3.866495,NaN,NaN,NaN,NaN,NaN,NaN
122,2025-07-31,A000660,3.621530,NaN,NaN,NaN,NaN,NaN,NaN
123,2025-08-31,A000660,3.283688,NaN,NaN,NaN,NaN,NaN,NaN
124,2025-09-30,A000660,4.241934,NaN,NaN,NaN,NaN,NaN,NaN
125,2025-10-31,A000660,6.933568,NaN,NaN,NaN,NaN,NaN,NaN
126,2025-11-30,A000660,NaN,6.907504,4.380324,3.447113,7.511351,7.386383,6.940047
127,2025-12-31,A000660,NaN,7.071749,4.670657,3.680938,7.701076,7.092221,6.947077
128,2026-01-31,A000660,NaN,7.159487,4.787485,3.793189,7.800710,7.189793,6.954107


In [47]:
mc_long = get_series_by_keyword(ticker="A000660", forecast_date="2025-10-31", keyword="mc_", wide=False)

# ex_long → indicator를 컬럼으로 피벗
mc_pivot = (
    mc_long
    .pivot_table(
        index=["date", "ticker"],      # 행 인덱스
        columns="indicator",           # 열로 변환할 컬럼
        values="value",                # 값으로 쓸 컬럼
        aggfunc="last"                 # 중복 시 마지막 값 사용
    )
    .reset_index()                     # date, ticker를 일반 컬럼으로 되돌림
)

# 필요 시 indicator 컬럼명 정리 (MultiIndex 제거)
mc_pivot.columns.name = None

# 확인
print(mc_pivot.head())

        date   ticker        mc_ets       mc_lstm    mc_prophet  \
0 2025-11-30  A000660  5.731205e+11  3.197221e+11  2.584028e+11   
1 2025-12-31  A000660  6.149089e+11  3.133025e+11  2.606100e+11   
2 2026-01-31  A000660  6.225380e+11  3.211392e+11  2.685574e+11   
3 2026-02-28  A000660  6.073233e+11  3.221803e+11  2.516488e+11   
4 2026-03-31  A000660  6.514963e+11  3.031929e+11  2.502522e+11   

   mc_sarima_exog  mc_sarima_noexog      mc_theta  
0    6.169891e+11      6.027350e+11  5.691450e+11  
1    6.719847e+11      5.893380e+11  5.887775e+11  
2    6.806786e+11      5.974458e+11  5.893733e+11  
3    6.615186e+11      5.792403e+11  5.899692e+11  
4    7.150219e+11      5.872156e+11  6.254775e+11  


In [48]:
mc_pivot.tail(14)

,date,ticker,mc_ets,mc_lstm,mc_prophet,mc_sarima_exog,mc_sarima_noexog,mc_theta
0,2025-11-30,A000660,5.731205e+11,3.197221e+11,2.584028e+11,6.169891e+11,6.027350e+11,5.691450e+11
1,2025-12-31,A000660,6.149089e+11,3.133025e+11,2.606100e+11,6.719847e+11,5.893380e+11,5.887775e+11
2,2026-01-31,A000660,6.225380e+11,3.211392e+11,2.685574e+11,6.806786e+11,5.974458e+11,5.893733e+11
3,2026-02-28,A000660,6.073233e+11,3.221803e+11,2.516488e+11,6.615186e+11,5.792403e+11,5.899692e+11
4,2026-03-31,A000660,6.514963e+11,3.031929e+11,2.502522e+11,7.150219e+11,5.872156e+11,6.254775e+11
5,2026-04-30,A000660,6.493821e+11,2.960346e+11,2.462231e+11,7.073560e+11,5.805011e+11,6.261086e+11
6,2026-05-31,A000660,6.550695e+11,2.853017e+11,2.589213e+11,7.197809e+11,5.916485e+11,6.267396e+11
7,2026-06-30,A000660,6.890985e+11,2.430872e+11,2.583401e+11,8.167877e+11,6.018101e+11,6.312626e+11
8,2026-07-31,A000660,6.777638e+11,2.313812e+11,2.458080e+11,7.961081e+11,5.862415e+11,6.318976e+11
9,2026-08-31,A000660,6.614447e+11,2.170572e+11,2.291188e+11,7.690963e+11,5.647760e+11,6.325326e+11


In [23]:
tickers = get_unique_tickers_by_forecast_date(
    db_info=db_info,
    forecast_date=None,  # 예: NULL rows
    table_name="Korea_company_valuation_ver2",
    use_date_only=True,
)
print("[Unique tickers]\n", tickers.head())

[Unique tickers]
 0    A000270
1    A000500
2    A000660
3    A001440
4    A002350
Name: ticker, dtype: object


In [24]:
tickers

0      A000270
1      A000500
2      A000660
3      A001440
4      A002350
5      A004000
6      A005380
7      A005930
8      A006400
9      A006910
10     A007700
11     A009150
12     A010120
13     A010140
14     A011780
15     A031980
16     A033500
17     A035420
18     A035720
19     A036190
20     A042370
21     A042660
22    A042700 
23     A043150
24     A044820
25     A051910
26     A059090
27     A060980
28     A068270
29     A071280
30     A071970
31     A073240
32     A077360
33     A082740
34     A084370
35     A086390
36     A093520
37     A095610
38     A103140
39     A103590
40     A105630
41     A114810
42     A123330
43     A123700
44     A131290
45     A140860
46     A161390
47     A207940
48     A214150
49     A232140
50     A253590
51     A375500
Name: ticker, dtype: object

In [52]:
# 2) 변화율 표 (예: forecast_date가 NULL, 모든 indicator, 2027-03-31 → 2027-06-30)
df_change = get_indicator_change_rates(
    db_info=db_info,
    forecast_date= "2025-10-31",                    # 또는 '2027-06-30' 등
    indicator= 'mc_sarima_exog',                        # 'all' 또는 ['sarima_forecast','lstm_forecast'] 등
    date1="2025-11-30",
    date2="2026-11-30",
    table_name="Korea_company_valuation_ver2",
    use_date_only_for_forecast=True,
    drop_zero_base=True,
)
print("\n[Change table]\n", df_change.head(20))


[Change table]
 d        indicator   ticker       date1       date2   value_date1  \
0   mc_sarima_exog  A006910  2025-11-30  2026-11-30  4.484007e+08   
1   mc_sarima_exog  A298040  2025-11-30  2026-11-30  3.047304e+10   
2   mc_sarima_exog  A042660  2025-11-30  2026-11-30  6.502481e+10   
3   mc_sarima_exog  A000660  2025-11-30  2026-11-30  6.169891e+11   
4   mc_sarima_exog  A033500  2025-11-30  2026-11-30  1.318210e+09   
5   mc_sarima_exog  A103140  2025-11-30  2026-11-30  4.063954e+09   
6   mc_sarima_exog  A010140  2025-11-30  2026-11-30  3.634435e+10   
7   mc_sarima_exog  A098120  2025-11-30  2026-11-30  3.016861e+08   
8   mc_sarima_exog  A042700  2025-11-30  2026-11-30  2.308461e+10   
9   mc_sarima_exog  A082740  2025-11-30  2026-11-30  5.554234e+09   
10  mc_sarima_exog  A131970  2025-11-30  2026-11-30  1.339920e+09   
11  mc_sarima_exog  A010120  2025-11-30  2026-11-30  2.081106e+10   
12  mc_sarima_exog  A114810  2025-11-30  2026-11-30  5.131411e+08   
13  mc_sarima_exo